## Pre-Trajectory Sampling with Batch Execution (PTSBE)

PTSBE (Pre-Trajectory Sampling with Batch Execution) is a method for sampling from noisy quantum circuits efficiently. Instead of simulating the full density matrix and sampling once per shot, PTSBE:

1. Traces the kernel to get the gate sequence and qubit layout.
2. Extracts noise sites by matching the noise model to the trace (each noisy gate becomes a *noise site* with a set of Kraus outcomes, e.g. $I$, $X$, $Y$, $Z$ for depolarization).
3. Generates trajectories — each trajectory is one possible *realization* of noise (one Kraus outcome per site). A *sampling strategy* decides which trajectories to use (e.g. by sampling trajectories proportional to their likelihood).
4. Allocates shots across trajectories (e.g. by error likelihood).
5. Runs batches — for each trajectory, the circuit is run as a noiseless circuit with that trajectory's outcomes applied; results are collected.
6. Aggregates all per-trajectory counts into a single `SampleResult`.

Given sufficient trajectory and shot samples PTSBE will be equivalent to standard trajectory based sampling up to sampling noise. However, it has the additional advantage of being able to batch many shots per trajectory and to control simulation cost via the number of trajectories. This notebook runs the full workflow with a single API call, `cudaq.ptsbe.sample()`, on one running example: a noisy GHZ circuit that grows from a first sampling call to trajectory inspection, a performance comparison, and finally a mid-circuit parity check.

### Set up the environment

In [1]:
import cudaq

cudaq.set_target("nvidia")
cudaq.set_random_seed(42)

### Define the circuit and noise model

The running example is GHZ state preparation on `n` qubits: a Hadamard followed by a chain of $CNOT$ gates. Attach a noise model to make it noisy. Each gate you add to the noise model becomes a noise site when that gate appears in the circuit. For the single-qubit $H$ we use a `DepolarizationChannel` with one qubit, and for $CNOT$ we add `Depolarization2` (two-qubit depolarization channel) with `num_controls=1` so it applies to the **qubit pair** `[control, target]`.

With a kernel and a noise model in hand, we can sample.

In [2]:
depol_1q = 0.01
depol_2q = 0.01

@cudaq.kernel
def ghz(n: int):
    q = cudaq.qvector(n)
    h(q[0])
    for i in range(1, n):
        x.ctrl(q[i - 1], q[i])
    mz(q)

noise = cudaq.NoiseModel()
noise.add_all_qubit_channel("h", cudaq.DepolarizationChannel(depol_1q))
noise.add_all_qubit_channel("x", cudaq.Depolarization2(depol_2q), num_controls=1)

### Run PTSBE sampling

Call `cudaq.ptsbe.sample()` with the kernel, its arguments, the noise model, and a shot count. The defaults take care of everything else: trajectories are drawn from the noise model and shots are allocated across them automatically. For reference we also run standard noisy sampling with `cudaq.sample()`, which simulates one trajectory per shot. The two distributions agree up to sampling noise.

In [3]:
shots = 10000
n_qubits = 4

result = cudaq.ptsbe.sample(ghz, n_qubits, noise_model=noise, shots_count=shots)
print("PTSBE:   ", result)

result_standard = cudaq.sample(ghz, n_qubits, noise_model=noise, shots_count=shots)
print("Standard:", result_standard)

PTSBE:    { 0000:4974 0001:12 0010:12 0011:30 0100:12 0111:49 1000:41 1011:9 1100:22 1101:10 1110:15 1111:4814 }

Standard: { 0000:4846 0001:9 0010:14 0011:28 0100:17 0111:32 1000:30 1010:1 1011:8 1100:30 1101:14 1110:7 1111:4964 }



#### Inline noise with `apply_noise`

Instead of (or in addition to) attaching noise via a `NoiseModel`, you can place noise at specific points in the kernel with `cudaq.apply_noise`. PTSBE traces these calls and includes them as noise sites alongside any model-attached noise. Here the same GHZ circuit carries its noise inline, so no `noise_model` argument is needed.

However the noise is specified, PTSBE turns it into a space of trajectories. The next section shows how to control which trajectories are simulated and how shots are spread across them.

In [4]:
@cudaq.kernel
def ghz_inline_noise(n: int):
    q = cudaq.qvector(n)
    h(q[0])
    cudaq.apply_noise(cudaq.DepolarizationChannel, depol_1q, q[0])
    for i in range(1, n):
        x.ctrl(q[i - 1], q[i])
        cudaq.apply_noise(cudaq.Depolarization2, depol_2q, q[i - 1], q[i])
    mz(q)

result_inline = cudaq.ptsbe.sample(ghz_inline_noise, n_qubits, shots_count=shots)
print("Inline apply_noise result:")
print(result_inline)

Inline apply_noise result:
{ 0000:4828 0001:13 0010:18 0011:22 0100:9 0111:46 1000:44 1011:15 1100:27 1101:9 1110:9 1111:4960 }



### Controlling trajectories and shot allocation

Every call so far used the default trajectory machinery. Three optional arguments expose it:

- `max_trajectories`: cap the number of distinct trajectories that are simulated, which bounds simulation cost independently of the shot count.
- `sampling_strategy` — how trajectories are chosen:
  - `ProbabilisticSamplingStrategy(seed=...)` (default): Performs Monte Carlo sampling over trajectories.
  - `ExhaustiveSamplingStrategy()`: use all possible trajectories (every combination of Kraus outcomes per noise site).
  - `OrderedSamplingStrategy()`: Select the top-$k$ trajectories by probability (highest first), up to `max_trajectories`.
- `shot_allocation` — how shots are split across the chosen trajectories:
  - `PROPORTIONAL` (default): allocate shots in proportion to each sampled trajectory's weighting.
  - `UNIFORM`: give each trajectory the same number of shots.
  - `LOW_WEIGHT_BIAS`: bias more shots toward low-weight (fewer errors) trajectories, optional `bias_strength` (default 2.0).
  - `HIGH_WEIGHT_BIAS`: bias more shots toward high-weight trajectories, optional `bias_strength` (default 2.0).

This example runs `OrderedSamplingStrategy` with 16 trajectories, once with `LOW_WEIGHT_BIAS` shot allocation to emphasize likely low-error trajectories and once with the default `PROPORTIONAL` allocation.

Capping trajectories raises a question: which noise realizations were actually simulated, and how many shots did each receive? Execution data answers that.

In [5]:
strategy = cudaq.ptsbe.OrderedSamplingStrategy()
max_traj = 16

def print_result(label, sample_result):
    print(f"{label:<36} shots={sample_result.get_total_shots():>6}  counts={sample_result}")

biased_alloc = cudaq.ptsbe.ShotAllocationStrategy(
    cudaq.ptsbe.ShotAllocationType.LOW_WEIGHT_BIAS,
    bias_strength=3.0,
    seed=42,
)
result_biased = cudaq.ptsbe.sample(
    ghz,
    n_qubits,
    noise_model=noise,
    shots_count=shots,
    sampling_strategy=strategy,
    shot_allocation=biased_alloc,
    max_trajectories=max_traj,
)

unbiased_alloc = cudaq.ptsbe.ShotAllocationStrategy(
    cudaq.ptsbe.ShotAllocationType.PROPORTIONAL,
    seed=42,
)
result_unbiased = cudaq.ptsbe.sample(
    ghz,
    n_qubits,
    noise_model=noise,
    shots_count=shots,
    sampling_strategy=strategy,
    shot_allocation=unbiased_alloc,
    max_trajectories=max_traj,
)

print(f"Comparison with same shots={shots}, max_trajectories={max_traj}")
print_result("PTSBE (ordered + low-weight bias)", result_biased)
print_result("PTSBE (ordered + proportional)", result_unbiased)

Comparison with same shots=10000, max_trajectories=16
PTSBE (ordered + low-weight bias)    shots= 10000  counts={ 0000:4932 0111:4 1000:1 1111:5063 }

PTSBE (ordered + proportional)       shots= 10000  counts={ 0000:4980 0111:17 1000:18 1100:4 1111:4981 }



### Inspecting trajectories with execution data

To make the trajectory space interesting, grow the same GHZ circuit to 12 qubits. With depolarization on every gate this creates 12 noise sites and a large trajectory space.

Pass `return_execution_data=True` to get the PTSBE execution data via `result.ptsbe_execution_data`. This reveals *why* PTSBE is efficient: most probability mass concentrates on a few low-error trajectories, so a small number of circuit simulations captures the bulk of the physics. The highest-probability trajectory (no errors at any noise site) typically receives the vast majority of shots, while multi-error trajectories are exponentially suppressed.

Note: this is an experimental API and may change in future releases.

Concentrating shots on a handful of trajectories is exactly what makes PTSBE fast. The next section measures the effect.

In [6]:
from collections import Counter

n_qubits = 12
exec_shots = 1_000_000
max_traj = 256
result_with_data = cudaq.ptsbe.sample(
    ghz,
    n_qubits,
    noise_model=noise,
    shots_count=exec_shots,
    max_trajectories=max_traj,
    return_execution_data=True,
)
assert result_with_data.has_execution_data()
data = result_with_data.ptsbe_execution_data

trajs = sorted(data.trajectories, key=lambda t: t.probability, reverse=True)
print(f"Trajectories: {len(trajs)}, Total shots: {exec_shots:,}\n")

# `t.probability` is the per-shot probability mass in the full trajectory distribution.
print("Top 5 trajectories (highest probability):")
cumulative = 0
for rank, t in enumerate(trajs[:5], 1):
    cumulative += t.num_shots
    n_errors = sum(1 for s in t.kraus_selections if s.is_error)
    p_theory = t.probability
    p_empirical = t.num_shots / exec_shots
    print(f"  #{rank}: p_theory={p_theory:.6f}, p_empirical={p_empirical:.6f}, "
          f"shots={t.num_shots:,}, errors={n_errors}, "
          f"cumulative shots={100 * cumulative / exec_shots:.1f}%")

# Lowest-probability trajectory
lowest = trajs[-1]
n_errors_low = sum(1 for s in lowest.kraus_selections if s.is_error)
print(f"\nLowest-probability trajectory:")
print(f"  prob={lowest.probability:.2e}, shots={lowest.num_shots}, errors={n_errors_low}")

noise_instructions = {i: inst for i, inst in enumerate(data.instructions)
                      if inst.type == cudaq.ptsbe.TraceInstructionType.Noise}

def fmt_selection(sel):
    # `channel.params` carries the noise rates passed to the channel, and
    # `channel.channel` is the underlying `cudaq.KrausChannel` with `.noise_type`,
    # `.parameters`, and `.get_ops()` for the Kraus matrices.
    channel = noise_instructions[sel.circuit_location]
    label = "error" if sel.is_error else "no-error"
    params = ", ".join(f"{p:g}" for p in channel.params)
    n_ops = len(channel.channel.get_ops())
    return (f"    site {sel.circuit_location} [{channel.name}(p={params}) on q{channel.targets}]: "
            f"K{sel.kraus_operator_index} / {n_ops} ops ({label})")

highest = trajs[0]
print(f"\nHighest-probability trajectory (id={highest.trajectory_id}):")
print(f"  prob={highest.probability:.6f}, shots={highest.num_shots:,}")
print(f"  Kraus selections:")
for sel in highest.kraus_selections:
    print(fmt_selection(sel))

print(f"\nLowest-probability trajectory (id={lowest.trajectory_id}):")
print(f"  prob={lowest.probability:.2e}, shots={lowest.num_shots}")
print(f"  Kraus selections:")
for sel in lowest.kraus_selections:
    print(fmt_selection(sel))

# Error count histogram
error_counts = Counter(
    sum(1 for s in t.kraus_selections if s.is_error) for t in trajs
)
print("\nTrajectories grouped by error count:")
for n_err in sorted(error_counts):
    n_traj = error_counts[n_err]
    total = sum(t.num_shots for t in trajs
                if sum(1 for s in t.kraus_selections if s.is_error) == n_err)
    print(f"  {n_err} errors: {n_traj} trajectories, "
          f"{total:,} shots ({100 * total / exec_shots:.1f}%)")

Trajectories: 147, Total shots: 1,000,000
[2026-07-12 15:31:16.210] [warning] [PTSBESampleResult.cpp:20] PTSBE execution data API is experimental and may change in a future release.

Top 5 trajectories (highest probability):
  #1: p_theory=0.886385, p_empirical=0.888297, shots=888,297, errors=0, cumulative shots=88.8%
  #2: p_theory=0.002984, p_empirical=0.002378, shots=2,378, errors=1, cumulative shots=89.1%
  #3: p_theory=0.002984, p_empirical=0.002333, shots=2,333, errors=1, cumulative shots=89.3%
  #4: p_theory=0.002984, p_empirical=0.004269, shots=4,269, errors=1, cumulative shots=89.7%
  #5: p_theory=0.000597, p_empirical=0.000434, shots=434, errors=1, cumulative shots=89.8%

Lowest-probability trajectory:
  prob=2.71e-10, shots=359, errors=3

Highest-probability trajectory (id=0):
  prob=0.886385, shots=888,297
  Kraus selections:
    site 1 [depolarization_channel(p=0.01) on q[0]]: K0 / 4 ops (no-error)
    site 3 [depolarization2(p=0.01) on q[1, 0]]: K0 / 16 ops (no-error)
   

### Performance of PTSBE vs standard noisy sampling

The table below compares the runtime of standard trajectory sampling with PTSBE on the 12-qubit GHZ circuit at increasing shot counts. Standard sampling simulates one trajectory per shot, so its cost grows with the shot count. PTSBE replays at most `max_trajectories` circuits no matter how many shots are requested.

So far the circuit only measures at the end. Real workloads, such as QEC experiments, measure and reset qubits in the middle of the circuit, which is where we take the example next.

In [7]:
import time

shot_counts = [1_000, 10_000, 100_000, 1_000_000]
max_traj = 64

print(f"{'shots':>10}  {'standard (s)':>14}  {'PTSBE (s)':>12}  {'speedup':>8}")
print("-" * 52)

for shots in shot_counts:
    # Standard noisy sampling (one trajectory per shot)
    t0 = time.perf_counter()
    cudaq.sample(ghz, n_qubits, noise_model=noise, shots_count=shots)
    t_standard = time.perf_counter() - t0

    # PTSBE (batched trajectories)
    t0 = time.perf_counter()
    cudaq.ptsbe.sample(
        ghz, n_qubits,
        noise_model=noise,
        shots_count=shots,
        max_trajectories=max_traj,
    )
    t_ptsbe = time.perf_counter() - t0

    speedup = t_standard / t_ptsbe if t_ptsbe > 0 else float('inf')
    print(f"{shots:>10,}  {t_standard:>14.3f}  {t_ptsbe:>12.3f}  {speedup:>7.1f}x")

     shots    standard (s)     PTSBE (s)   speedup
----------------------------------------------------
     1,000           0.008         0.007      1.2x


    10,000           0.098         0.009     11.0x


   100,000           0.715         0.018     39.1x


 1,000,000           5.752         0.036    158.4x


### Mid-circuit measurement and reset

PTSBE also supports kernels that measure and reset qubits mid-circuit. During replay of each sampled noise trajectory, every measurement site draws its outcome at the true conditional probability, collapses the state, records the bit, and applies any fused reset before the circuit continues.

To demonstrate, extend the GHZ circuit with an ancilla parity check, the measure-and-reset pattern used in QEC memory experiments: an ancilla accumulates the data parity $q_0 \oplus q_1$, is measured mid-circuit into the named register `syndrome`, and is reset for reuse. In a noiseless GHZ state the two data qubits always agree, so the mid-circuit syndrome bit fires only when noise flips a data qubit before extraction.

Two options become relevant only now:

- `include_sequential_data=True`: the result carries one bitstring record per shot, available through `result.get_sequential_data()`. It is forced on automatically when the kernel contains mid-circuit measurement or reset, because aggregate counts alone cannot represent per-shot measurement histories.
- `max_shots_per_path`: cap on how many shots share one replay of a trajectory. `None` (the default) selects 1 when the kernel contains mid-circuit measurement, so each shot draws its own independent measurement history, and unlimited otherwise. The environment variable `CUDAQ_PTSBE_MAX_SHOTS_PER_PATH` takes precedence.

Every measurement site (mid-circuit or terminal) contributes one bit to a fixed-width per-shot record:

- Each `result.get_sequential_data()` string is one full record in record-index order.
- `result.record_layout` describes the record: one `RecordSite` per bit, with `record_index`, `qubit`, `resets` (the measurement is fused with a following reset), `terminal`, and `register_name` (the kernel's measurement variable name, when named).
- The counts distribution is over full records rather than terminal bits alone.

In [8]:
import numpy as np

@cudaq.kernel
def ghz_with_parity_check(n: int):
    q = cudaq.qvector(n + 1)
    h(q[0])
    for i in range(1, n):
        x.ctrl(q[i - 1], q[i])
    x.ctrl(q[0], q[n])
    x.ctrl(q[1], q[n])
    syndrome = mz(q[n])
    reset(q[n])
    mz(q)

n_data = 4
mcm_result = cudaq.ptsbe.sample(
    ghz_with_parity_check,
    n_data,
    noise_model=noise,
    shots_count=10_000,
    include_sequential_data=True,
)

print("Record layout:")
for site in mcm_result.record_layout:
    print(f"  [{site.record_index}] qubit={site.qubit}  resets={site.resets}  "
          f"terminal={site.terminal}  register_name={site.register_name}")

records = np.array([[int(bit) for bit in record]
                    for record in mcm_result.get_sequential_data()],
                   dtype=np.uint8)
syndrome_col = next(s.record_index for s in mcm_result.record_layout
                    if s.resets)
ancilla_terminal_col = max(s.record_index for s in mcm_result.record_layout
                           if s.terminal and s.qubit == n_data)

print(f"\nRecord array shape (shots, bits): {records.shape}")
print(f"Mid-circuit syndrome rate: {records[:, syndrome_col].mean():.4f}")
print(f"Terminal ancilla reads 0 after reset: "
      f"{bool((records[:, ancilla_terminal_col] == 0).all())}")

Record layout:
  [0] qubit=4  resets=True  terminal=False  register_name=syndrome
  [1] qubit=0  resets=False  terminal=True  register_name=None
  [2] qubit=1  resets=False  terminal=True  register_name=None
  [3] qubit=2  resets=False  terminal=True  register_name=None
  [4] qubit=3  resets=False  terminal=True  register_name=None
  [5] qubit=4  resets=False  terminal=True  register_name=None

Record array shape (shots, bits): (10000, 6)
Mid-circuit syndrome rate: 0.0194
Terminal ancilla reads 0 after reset: True


### Appendix: `cudaq.ptsbe.sample` options reference

| Option | Default | Purpose |
| --- | --- | --- |
| `sampling_strategy` | `ProbabilisticSamplingStrategy` | How trajectories are chosen: Monte Carlo sampling, `ExhaustiveSamplingStrategy` (all combinations), or `OrderedSamplingStrategy` (top-$k$ by probability). |
| `shot_allocation` | `PROPORTIONAL` | How shots are split across trajectories: `PROPORTIONAL`, `UNIFORM`, `LOW_WEIGHT_BIAS`, or `HIGH_WEIGHT_BIAS` (biased modes take an optional `bias_strength`, default 2.0). |
| `max_trajectories` | `None` (unlimited) | Cap the number of trajectories, useful for large shot counts. |
| `include_sequential_data` | `False` | Carry one bitstring record per shot via `result.get_sequential_data()`. Forced on for kernels with mid-circuit measurement or reset. |
| `max_shots_per_path` | `None` | Cap on shots sharing one replay of a trajectory. `None` selects 1 with mid-circuit measurement, unlimited otherwise. `CUDAQ_PTSBE_MAX_SHOTS_PER_PATH` takes precedence. |
| `return_execution_data` | `False` | Include trace instructions and per-trajectory data via `result.ptsbe_execution_data`. Experimental and subject to change. |